In [8]:
import talib
import pandas as pd
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots
import yfinance as yf
import plotly.express as px
from sklearn.linear_model import LinearRegression
from datetime import datetime, timedelta
import warnings


### Анализ отношений рынков

In [9]:
#загрузим 10 ление данные по золоту и sp500
gold_df = yf.download('GC=F', period='10y', interval='1mo')
sp500_df = yf.download('^GSPC', period='10y', interval='1mo')
gold_df = gold_df.reset_index()

# Объединяем sp500_df с gold_df по столбцу 'Date', добавляя суффикс '_gold' для столбца 'Close' из gold_df
sp500_df = sp500_df.merge(
    gold_df[['Date', 'Close']], 
    on='Date', 
    how='left', 
    suffixes=('', '_gold')
)

# Заполняем пропущенные значения в столбце 'Close_gold' с использованием предыдущих значений
sp500_df['Close_gold'] = sp500_df['Close_gold'].ffill()

# посчитае gold/sp500 ratio
sp500_df["gsp_ratio"] = sp500_df['Close_gold'] / sp500_df["Close"].values
sp500_df["gsp_ratioSMA"] = talib.SMA(sp500_df["gsp_ratio"], timeperiod=12)
sp500_df.dropna(inplace=True)
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=sp500_df["Date"], y=sp500_df["gsp_ratio"], mode='lines', name='gold/sp500 ratio'), secondary_y=False)
fig.add_trace(go.Scatter(x=sp500_df["Date"], y=sp500_df["gsp_ratioSMA"], mode='lines', name='sma_ratio'), secondary_y=False)
fig.add_trace(go.Scatter(x=sp500_df["Date"], y=sp500_df["Close"], mode='markers+lines', name='SP500 Price'), secondary_y=True)

# Настроим оси
fig.update_layout(
    title='GOLD to SP500 Ratio vs SP500 Price (Last 10 Years)',
    yaxis=dict(title='Ratio GOLD to SP500'),
    yaxis2=dict(title='SP500 Price')
)

fig.show()

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


In [3]:
gold_df["usd_gold_ratio"] = 1/gold_df["Close"]
gold_df["SMA_us_gold_ratio"] = talib.SMA(gold_df["usd_gold_ratio"], timeperiod=12)
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=gold_df["Date"], y=gold_df["usd_gold_ratio"], mode='lines', name='usd/gold ratio'), secondary_y=False)
fig.add_trace(go.Scatter(x=gold_df["Date"], y=gold_df["SMA_us_gold_ratio"], mode='lines', name='sma_ratio'), secondary_y=False)
fig.add_trace(go.Scatter(x=gold_df["Date"], y=gold_df["Close"], mode='markers+lines', name='GOLD Price'), secondary_y=True)

# Настроим оси
fig.update_layout(
    title='USD to GOLD Ratio vs GOLD Price (Last 10 Years)',
    yaxis=dict(title='Ratio USD to GOLD'),
    yaxis2=dict(title='GOLD Price')
)

fig.show()

### Определение Относительной силы акции

In [10]:
# Пример определения RS
warnings.filterwarnings("ignore")

def calculate_alpha_beta(stock_data, sp500_data):
    stock_returns = stock_data.pct_change().dropna()
    sp500_returns = sp500_data.pct_change().dropna()
    
    df = pd.DataFrame({'stock': stock_returns, 'sp500': sp500_returns}).dropna()
    
    X = df['sp500'].values.reshape(-1, 1)
    y = df['stock'].values
    reg = LinearRegression().fit(X, y) # добавляем данные в модель
    
    beta = reg.coef_[0]
    alpha = reg.intercept_ 
    
    return alpha, beta

# Определим периоды
end_date = datetime.now()
train_start = end_date - timedelta(days=365*2)  # 1 год для обучения
test_start = end_date - timedelta(days=365)  # 1 год для тестирования

# Список акций для анализа
stocks = ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'NVDA', 'TSLA', 'JPM', 'JNJ', 'V', 
          'PG', 'UNH', 'HD', 'MA', 'DIS', 'ADBE', 'CRM', 'NFLX', 'PYPL', 'INTC']

# Загрузка данных S&P 500 и акций
data = yf.download(stocks + ['^GSPC'], start=train_start, end=end_date)
sp500 = data['Close']['^GSPC']

results = {}

for stock in stocks:
    try:
        stock_data = data['Close'][stock]
        
        # Разделение данных на обучающий и тестовый периоды
        train_stock_data = stock_data[stock_data.index < test_start]
        print(stock)
        print(f"Train Period : {train_stock_data.index[0]} - {train_stock_data.index[-1]} ")
        print(f"Test Period : {stock_data.loc[test_start:].index[0]} - {stock_data.loc[test_start:].index[-1]}")
       
        train_sp500_data = sp500[sp500.index < test_start]
        alpha, beta = calculate_alpha_beta(train_stock_data, train_sp500_data)
        
        # Расчет доходности на тестовом периоде
        test_return = (stock_data.loc[test_start:].iloc[-1] / stock_data.loc[test_start:].iloc[0]) - 1
        
        results[stock] = {'Alpha': alpha, 'Beta': beta, 'Test_Return': test_return}
    except Exception as e:
        print(f"Ошибка при обработке {stock}: {e}")

# Создание DataFrame с результатами
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('Alpha', ascending=False)

# Визуализация результатов через Plotly
fig = px.scatter(results_df, x='Alpha', y='Test_Return', text=results_df.index,
                 title='Alpha vs Actual Return', labels={'Alpha': 'Alpha (Training Period)', 'Test_Return': 'Return (Test Period)'})
fig.update_traces(textposition='top center')
fig.update_layout(showlegend=False)
fig.show()

# График цен на тестовом периоде для топ-5 и нижних-5 акций по альфе через Plotly
top_5 = results_df.head().index
bottom_5 = results_df.tail().index


fig = px.line()

for stock in top_5:
    stock_data = data['Close'][stock][test_start:]
    fig.add_scatter(x=stock_data.index, y=stock_data / stock_data.iloc[0], mode='lines', name=stock)

for stock in bottom_5:
    stock_data = data['Close'][stock][test_start:]
    fig.add_scatter(x=stock_data.index, y=stock_data / stock_data.iloc[0], mode='lines', name=stock, line=dict(dash='dash'))

fig.update_layout(title='Price Performance during Test Period', xaxis_title='Date', yaxis_title='Normalized Price')
fig.show()


print("Топ-5 лучших акций:")
print(top_5)

print("\nТоп-5 худших акций:")
print(bottom_5)


[*********************100%%**********************]  21 of 21 completed


AAPL
Train Period : 2022-09-06 00:00:00 - 2023-09-01 00:00:00 
Test Period : 2023-09-05 00:00:00 - 2024-08-30 00:00:00
MSFT
Train Period : 2022-09-06 00:00:00 - 2023-09-01 00:00:00 
Test Period : 2023-09-05 00:00:00 - 2024-08-30 00:00:00
AMZN
Train Period : 2022-09-06 00:00:00 - 2023-09-01 00:00:00 
Test Period : 2023-09-05 00:00:00 - 2024-08-30 00:00:00
GOOGL
Train Period : 2022-09-06 00:00:00 - 2023-09-01 00:00:00 
Test Period : 2023-09-05 00:00:00 - 2024-08-30 00:00:00
META
Train Period : 2022-09-06 00:00:00 - 2023-09-01 00:00:00 
Test Period : 2023-09-05 00:00:00 - 2024-08-30 00:00:00
NVDA
Train Period : 2022-09-06 00:00:00 - 2023-09-01 00:00:00 
Test Period : 2023-09-05 00:00:00 - 2024-08-30 00:00:00
TSLA
Train Period : 2022-09-06 00:00:00 - 2023-09-01 00:00:00 
Test Period : 2023-09-05 00:00:00 - 2024-08-30 00:00:00
JPM
Train Period : 2022-09-06 00:00:00 - 2023-09-01 00:00:00 
Test Period : 2023-09-05 00:00:00 - 2024-08-30 00:00:00
JNJ
Train Period : 2022-09-06 00:00:00 - 2023-09

Топ-5 лучших акций:
Index(['NVDA', 'NFLX', 'META', 'ADBE', 'CRM'], dtype='object')

Топ-5 худших акций:
Index(['AMZN', 'UNH', 'TSLA', 'DIS', 'PYPL'], dtype='object')


In [48]:
# Пример построения графиков 
# Функция для создания графика регрессии
def plot_regression(stock, stock_data, sp500_data):
    stock_returns = stock_data.pct_change().dropna()
    sp500_returns = sp500_data.pct_change().dropna()

    df = pd.DataFrame({'stock': stock_returns, 'sp500': sp500_returns}).dropna()

    X = df['sp500'].values.reshape(-1, 1)
    y = df['stock'].values
    reg = LinearRegression().fit(X, y)

    beta = reg.coef_[0]
    alpha = reg.intercept_  

    regression_line = reg.predict(X)

    fig = go.Figure()

    # Точки
    fig.add_trace(go.Scatter(x=sp500_returns, y=stock_returns,
                             mode='markers', name=f'{stock} Data'))

    # Линия регрессии
    fig.add_trace(go.Scatter(x=sp500_returns, y=regression_line,
                             mode='lines', name='Regression Line',
                             line=dict(color='gray')))

    fig.update_layout(title=f'{stock} Alpha: {alpha:.4f}, Beta: {beta:.4f}',
                      xaxis_title='S&P 500 Returns',
                      yaxis_title=f'{stock} Returns')

    fig.show()

# Построение графиков для топ-5 акций по альфе за весь период
for stock in top_5:
    stock_data = data['Close'][stock][train_start:end_date]
    plot_regression(stock, stock_data, sp500[train_start:end_date])